# Hamilton Vantage

The Hamilton Vantage is a liquid handler with independent pipetting channels, an optional 96-head, and an optional Integrated Plate Gripper (IPG). This notebook walks through the `Vantage` device API, covering setup, deck layout, tip pickup, aspiration and dispensing, IPG plate transport, and teardown.

**Prerequisites:**

- Installed PyLabRobot with USB support: `pip install pylabrobot[usb]`
- Platform-specific driver setup (libusb on Mac, Zadig on Windows) — see [the installation guide](../../_getting-started/installation)
- Connected the Vantage to your computer using the USB cable

All commands in this notebook are sent to a chatterbox driver that logs firmware strings to the console without touching real hardware. Remove `chatterbox=True` and supply a real USB address to run on an actual instrument.

## Setup

Import {class}`~pylabrobot.hamilton.liquid_handlers.vantage.vantage.Vantage` and {class}`~pylabrobot.resources.hamilton.vantage_decks.VantageDeck`. `Vantage` is the device class that owns the driver and exposes capabilities: `vantage.pip` (independent channels), `vantage.head96` (96-head, if installed), and `vantage.ipg` (plate gripper arm, if installed).

`VantageDeck` requires a `size` parameter. The standard 1.3 m configuration is used here.

In [ ]:
from pylabrobot.hamilton.liquid_handlers.vantage import Vantage
from pylabrobot.resources.hamilton import VantageDeck

deck = VantageDeck(size=1.3)
vantage = Vantage(deck=deck, chatterbox=True)

Call {meth}`~pylabrobot.hamilton.liquid_handlers.vantage.vantage.Vantage.setup` to initialise the driver and wire up the PIP, Head96, and IPG capability frontends.

In [ ]:
await vantage.setup()

print(f"pip channels  : {vantage.pip.num_channels}")
print(f"head96 present: {vantage.head96 is not None}")
print(f"ipg present   : {vantage.ipg is not None}")

## Creating the deck layout

Define the physical deck layout by assigning carriers with tip racks and plates. This tutorial uses:

- {class}`~pylabrobot.resources.hamilton.tip_carriers.TIP_CAR_480_A00` tip carrier
- {class}`~pylabrobot.resources.hamilton.plate_carriers.PLT_CAR_L5AC_A00` plate carrier
- {class}`~pylabrobot.resources.corning.plates.Cor_96_wellplate_360ul_Fb` 96-well plate
- {class}`~pylabrobot.resources.hamilton.tip_racks.hamilton_96_tiprack_1000uL_filter` tip rack

In [ ]:
from pylabrobot.resources import (
    Cor_96_wellplate_360ul_Fb,
    PLT_CAR_L5AC_A00,
    TIP_CAR_480_A00,
    hamilton_96_tiprack_1000uL_filter,
)

tip_car = TIP_CAR_480_A00(name="tip_carrier")
tip_car[0] = hamilton_96_tiprack_1000uL_filter(name="tips_01")
tip_car[1] = hamilton_96_tiprack_1000uL_filter(name="tips_02")
deck.assign_child_resource(tip_car, rails=3)

plt_car = PLT_CAR_L5AC_A00(name="plate_carrier")
plt_car[0] = Cor_96_wellplate_360ul_Fb(name="plate_01")
plt_car[1] = Cor_96_wellplate_360ul_Fb(name="plate_02")
deck.assign_child_resource(plt_car, rails=15)

Inspect the deck layout using {meth}`~pylabrobot.resources.deck.Deck.summary`.

In [ ]:
deck.summary()

## Picking up tips

`vantage.pip` is a {class}`~pylabrobot.capabilities.liquid_handling.pip.PIP` capability frontend. Its API is identical to `star.pip` on the STAR. Indexing a tip rack with `["A1:C1"]` returns the first three tip spots in column 1.

In [ ]:
tiprack = deck.get_resource("tips_01")
await vantage.pip.pick_up_tips(tiprack["A1:C1"])

## Aspirating and dispensing

Aspirate from wells `A1:C1` of the source plate and dispense into `A1:C1` of the destination plate, using channels 0, 1, and 2.

In [ ]:
src = deck.get_resource("plate_01")
dst = deck.get_resource("plate_02")

await vantage.pip.aspirate(src["A1:C1"], vols=[100.0, 50.0, 200.0])

In [ ]:
await vantage.pip.dispense(dst["A1:C1"], vols=[100.0, 50.0, 200.0])

Move the liquid back to the source plate:

In [ ]:
await vantage.pip.aspirate(dst["A1:C1"], vols=[100.0, 50.0, 200.0])
await vantage.pip.dispense(src["A1:C1"], vols=[100.0, 50.0, 200.0])

## Dropping tips

Return tips to their original positions:

In [ ]:
await vantage.pip.drop_tips(tiprack["A1:C1"])

## Backend parameters

For Vantage-specific tuning, pass `backend_params` to any PIP operation. The available parameter dataclasses live on `VantagePIPBackend` in `pylabrobot.hamilton.liquid_handlers.vantage.pip_backend`. For example, to aspirate with capacitive liquid level detection (cLLD, `lld_mode=1`) enabled on two channels:

In [ ]:
from pylabrobot.hamilton.liquid_handlers.vantage.pip_backend import VantagePIPBackend

tiprack2 = deck.get_resource("tips_02")
await vantage.pip.pick_up_tips(tiprack2["A1:B1"])

await vantage.pip.aspirate(
    src["A1:B1"],
    vols=[100.0, 100.0],
    backend_params=VantagePIPBackend.AspirateParams(
        lld_mode=[1, 1],  # 1 = cLLD (capacitive liquid level detection)
    ),
)

await vantage.pip.dispense(dst["A1:B1"], vols=[100.0, 100.0])
await vantage.pip.drop_tips(tiprack2["A1:B1"])

## IPG: Integrated Plate Gripper

`vantage.ipg` is an {class}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm` capability frontend backed by {class}`~pylabrobot.hamilton.liquid_handlers.vantage.ipg.IPGBackend`. It is `None` if the IPG is not installed on this configuration. The chatterbox driver always provides one.

Use {meth}`~pylabrobot.capabilities.arms.orientable_arm.OrientableArm.move_resource` to pick up a plate from one carrier slot and place it in another. Pass `pickup_backend_params` and `drop_backend_params` to override IPG-specific firmware parameters.

```{note}
`press_on_distance` on `IPGBackend.DropParams` is accepted for API compatibility but is not forwarded to the firmware: the `zi` parameter on the A1RM:DR command is uncharacterised on real hardware and has been left unsent until it can be verified safe.
```

```{warning}
`vantage.ipg.halt()`, `vantage.ipg.close_gripper()`, `vantage.ipg.is_gripper_closed()`, and `vantage.ipg.request_gripper_location()` raise `NotImplementedError` — they have not yet been ported from the legacy backend. Do not rely on `halt()` for emergency stop.
```

In [ ]:
if vantage.ipg is not None:
    plate = deck.get_resource("plate_01")
    destination_slot = plt_car[2]  # empty slot 2

    await vantage.ipg.move_resource(
        resource=plate,
        to=destination_slot,
        pickup_distance_from_top=13.2,
    )
else:
    print("IPG not present on this configuration — skipping.")

## Teardown

Call {meth}`~pylabrobot.hamilton.liquid_handlers.vantage.vantage.Vantage.stop` to stop all capability frontends in reverse order and close the driver connection.

In [ ]:
await vantage.stop()